# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
print("Libraries imported")

Libraries imported


## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [4]:
df= pd.read_csv('AviationData.csv', low_memory=False, encoding='latin-1')
df.columns = df.columns.str.strip().str.replace(' ', '.', regex=False).str.replace('_', '.', regex=False)

print("="*50)
print(f"DATAFRAME SHAPE: {df.shape[0]} rows, {df.shape[1]} columns")
print("="*50)

print("\n--- DATATYPES & ENTRY COUNTS ---")
print(df.info())

print("\n--- MISSING VALUES (NaNs) per column ---")
missing_data = df.isna().sum()
print(missing_data[missing_data > 0].sort_values(ascending=False))

print("\n--- NUMERICAL SUMMARY STATISTICS ---")
print(df.describe())

print("\n--- INSPECTION ---")
print("\nAircraft Damage Profile:")
print(df['Aircraft.damage'].value_counts(dropna=False).head())

print("\nAmateur Built Fleet Profile:")
print(df['Amateur.Built'].value_counts(dropna=False).head())




DATAFRAME SHAPE: 88889 rows, 31 columns

--- DATATYPES & ENTRY COUNTS ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  object 
 9   Airport.Name            52704 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Numb

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [10]:
df.columns = df.columns.str.strip().str.lower()

df = df[df['amateur.built'].astype(str).str.upper().str.strip() == 'NO']
df = df[df['aircraft.category'].astype(str).str.upper().str.strip() == 'AIRPLANE']

df['event.date'] = pd.to_datetime(df['event.date'], errors='coerce')
df = df.dropna(subset=['event.date'])

df = df[(df['event.date'].dt.year >= 1983) & (df['event.date'].dt.year <= 2023)]

print("=" * 50)
print("FILTERING COMPLETE")
print(f"Remaining active fleet records for analysis: {df.shape} rows")
print("=" * 50)


FILTERING COMPLETE
Remaining active fleet records for analysis: (21447, 31) rows


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [11]:
# List injury columns to fix
injury_cols = ['total.fatal.injuries', 'total.serious.injuries', 'total.minor.injuries', 'total.uninjured']

# Fill empty injury slots with 0 and turn floats into integers
for col in injury_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

# Add all groups together to get total people on the plane
df['total.aboard'] = df['total.fatal.injuries'] + df['total.serious.injuries'] + df['total.minor.injuries'] + df['total.uninjured']

# Combine fatal and serious counts for major injury total
df['fatal.or.serious'] = df['total.fatal.injuries'] + df['total.serious.injuries']

# Calculate injury rate as a percentage of total people aboard
df['injury.rate'] = np.where(df['total.aboard'] > 0, df['fatal.or.serious'] / df['total.aboard'], 0.0)

# Clean up text in the aircraft damage column
df['aircraft.damage'] = df['aircraft.damage'].astype(str).str.strip().str.upper()

# Create a 1 or 0 flag to easily count completely destroyed planes
df['hull.destroyed'] = np.where(df['aircraft.damage'] == 'DESTROYED', 1, 0)

# Check our work and show the final numbers
print("=" * 50)
print("METRIC ENGINEERING COMPLETE")
print(df[['total.aboard', 'injury.rate', 'hull.destroyed']].describe())
print("=" * 50)


METRIC ENGINEERING COMPLETE
       total.aboard   injury.rate  hull.destroyed
count  21447.000000  21447.000000    21447.000000
mean       8.592297      0.272002        0.107987
std       35.910084      0.426343        0.310371
min        0.000000      0.000000        0.000000
25%        1.000000      0.000000        0.000000
50%        2.000000      0.000000        0.000000
75%        2.000000      0.666667        0.000000
max      588.000000      1.000000        1.000000


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [12]:
df['aircraft.damage'] = df['aircraft.damage'].astype(str).str.strip().str.upper()
df['hull.destroyed'] = np.where(df['aircraft.damage'] == 'DESTROYED', 1, 0)
print("=" * 50)
print("HULL DESTRUCTION METRIC CONSTRUCTED")
print(df['hull.destroyed'].value_counts())
print("=" * 50)

HULL DESTRUCTION METRIC CONSTRUCTED
hull.destroyed
0    19131
1     2316
Name: count, dtype: int64


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [13]:
#1.  The raw dataset has messy entries with inconsistent casing and trailing punctuation (e.g., Boeing, BOEING, Boeing Company). We will convert all strings to uppercase and strip trailing whitespace to group them accurately.
#2.  To ensure our recommendations are statistically robust for the insurer, we will filter out rare manufacturers or custom experimental setups. We are setting a floor threshold of at least 50 documented incidents per manufacturer group.

df['make'] = df['make'].astype(str).str.strip().str.upper()
make_counts = df['make'].value_counts()
robust_makes = make_counts[make_counts >= 50].index
df = df[df['make'].isin(robust_makes)]
print("=" * 50)
print("MANUFACTURER CLEANING & ROBUSTNESS FILTER COMPLETE")
print(f"Remaining robust data shape: {df.shape}")
print("\nTop Remaining Manufacturers in Dataset:")
print(df['make'].value_counts().head(10))
print("=" * 50)


MANUFACTURER CLEANING & ROBUSTNESS FILTER COMPLETE
Remaining robust data shape: (17892, 35)

Top Remaining Manufacturers in Dataset:
make
CESSNA                7146
PIPER                 3989
BEECH                 1431
BOEING                1264
MOONEY                 363
AIRBUS                 243
CIRRUS DESIGN CORP     220
BELLANCA               219
AIR TRACTOR INC        219
MAULE                  215
Name: count, dtype: int64


### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [14]:
df = df.dropna(subset=['model'])
df['model'] = df['model'].astype(str).str.strip().str.upper()

df['make_model_id'] = df['make'] + "_" + df['model']

df.to_csv('Clean_Aviation_Data.csv', index=False)

print("=" * 50)
print("MODEL CLEANING & UNIQUE ID GENERATION COMPLETE")
print(f"Final clean dataset shape: {df.shape}")
print("\nSample of engineered Unique Aircraft Identifiers:")
print(df['make_model_id'].value_counts().head(10))
print("=" * 50)

MODEL CLEANING & UNIQUE ID GENERATION COMPLETE
Final clean dataset shape: (17879, 36)

Sample of engineered Unique Aircraft Identifiers:
make_model_id
CESSNA_172     769
BOEING_737     403
CESSNA_152     316
CESSNA_182     304
CESSNA_172S    276
PIPER_PA28     273
CESSNA_172N    249
CESSNA_180     213
CESSNA_172M    180
CESSNA_150     179
Name: count, dtype: int64


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [15]:
operational_cols = ['engine.type', 'weather.condition', 'purpose.of.flight', 'broad.phase.of.flight']

for col in operational_cols:
    df[col] = df[col].astype(str).str.strip().str.upper()

df['number.of.engines'] = pd.to_numeric(df['number.of.engines'], errors='coerce')

df.to_csv('Clean_Aviation_Data.csv', index=False)

print("=" * 50)
print("OPERATIONAL COLUMNS CLEANING COMPLETE")
print(f"Final processed dataset rows: {df.shape[0]}")
print("\nUnique Weather Conditions Found:")
print(df['weather.condition'].value_counts().head(5))
print("\nUnique Engine Counts Found:")
print(df['number.of.engines'].value_counts(dropna=False).head(5))
print("=" * 50)

OPERATIONAL COLUMNS CLEANING COMPLETE
Final processed dataset rows: 17879

Unique Weather Conditions Found:
weather.condition
VMC    14295
NAN     2417
IMC      905
UNK      262
Name: count, dtype: int64

Unique Engine Counts Found:
number.of.engines
1.0    13222
2.0     2470
NaN     2089
4.0       67
3.0       26
Name: count, dtype: int64


### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [16]:
nan_fractions = df.isna().mean()
excessive_nan_cols = nan_fractions[nan_fractions > 0.5].index

df = df.drop(columns=excessive_nan_cols)
df.to_csv('Clean_Aviation_Data.csv', index=False)

print("=" * 50)
print("COLUMN REMOVAL COMPLETE")
print(f"Dropped fields with >50% missing data: {list(excessive_nan_cols)}")
print(f"Final retained shape for underwriting analysis: {df.shape}")
print("=" * 50)


COLUMN REMOVAL COMPLETE
Dropped fields with >50% missing data: ['schedule', 'air.carrier']
Final retained shape for underwriting analysis: (17879, 34)


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [17]:
df.to_csv('Clean_Aviation_Data.csv', index=False)

print("=" * 50)
print("FINAL DATA SAVE COMPLETE")
print("Cleaned database exported as: 'Clean_Aviation_Data.csv'")
print("Data Cleaning Notebook is now finalized.")
print("=" * 50)

FINAL DATA SAVE COMPLETE
Cleaned database exported as: 'Clean_Aviation_Data.csv'
Data Cleaning Notebook is now finalized.
